In [9]:
import re
import numpy as np
import pandas as pd

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


# =========================================================
# 1) تحميل البيانات
# =========================================================
file_path = "14-761-cats-100-record.xlsx"
df = df.sample(n=1000, random_state=42)

df = pd.read_excel(file_path)

# التأكد من الأعمدة المطلوبة
required_columns = ["title", "content", "portalText"]
for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"العمود '{col}' غير موجود في الملف.")

# حذف الصفوف التي فيها قيم ناقصة
df = df[["content", "portalText"]].dropna().copy()
# تحويل القيم إلى نص
df["content"] = df["content"].astype(str)
df["portalText"] = df["portalText"].astype(str)

print("عدد السجلات بعد التنظيف:", len(df))
print("عدد الأصناف:", df["portalText"].nunique())


# =========================================================
# 2) تنظيف النصوص وتقطيعها إلى كلمات
# =========================================================
def clean_text(text):
    text = text.lower()
    
    # حذف الروابط
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # حذف الإيميلات
    text = re.sub(r"\S+@\S+", " ", text)
    
    # حذف الأرقام
    text = re.sub(r"\d+", " ", text)
    
    # الإبقاء على العربية والإنجليزية فقط
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z\s]", " ", text)
    
    # حذف المسافات الزائدة
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

def tokenize(text):
    text = clean_text(text)
    tokens = text.split()
    return tokens


df["tokens"] = df["content"].apply(tokenize)

# حذف السجلات التي أصبحت فارغة بعد التنظيف
df = df[df["tokens"].apply(len) > 0].copy()

print("عدد السجلات بعد حذف النصوص الفارغة:", len(df))


# =========================================================
# 3) تقسيم البيانات إلى تدريب واختبار
# =========================================================
X = df["tokens"]
y = df["portalText"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("عدد بيانات التدريب:", len(X_train))
print("عدد بيانات الاختبار:", len(X_test))


# =========================================================
# 4) تدريب Word2Vec باستخدام نصوص التدريب فقط
# =========================================================
w2v_model = Word2Vec(
    sentences=X_train.tolist(),
    vector_size=100,   # حجم الشعاع
    window=5,          # حجم النافذة
    min_count=1,       # أقل تكرار للكلمة
    workers=4,
    sg=1,              # 1 = Skip-gram, 0 = CBOW
    epochs=20
)

print("تم تدريب Word2Vec بنجاح.")


# =========================================================
# 5) تحويل كل نص إلى شعاع واحد
#    عبر متوسط أشعة الكلمات الموجودة فيه
# =========================================================
def document_vector(tokens, model):
    vectors = []
    
    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

X_train_vec = np.array([document_vector(tokens, w2v_model) for tokens in X_train])
X_test_vec  = np.array([document_vector(tokens, w2v_model) for tokens in X_test])

print("شكل أشعة التدريب:", X_train_vec.shape)
print("شكل أشعة الاختبار:", X_test_vec.shape)



عدد السجلات بعد التنظيف: 76100
عدد الأصناف: 759
عدد السجلات بعد حذف النصوص الفارغة: 76100
عدد بيانات التدريب: 60880
عدد بيانات الاختبار: 15220
تم تدريب Word2Vec بنجاح.
شكل أشعة التدريب: (60880, 100)
شكل أشعة الاختبار: (15220, 100)


In [10]:

# =========================================================
# 6) ترميز الأصناف
# =========================================================
label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print("الأصناف:", list(label_encoder.classes_))


# =========================================================
# 7) تدريب المصنف
# =========================================================
clf = LogisticRegression(
    max_iter=2000,
    random_state=42
)

clf.fit(X_train_vec, y_train_enc)

print("تم تدريب المصنف بنجاح.")



الأصناف: ['2=الأساطير الأوقيانوسية', '3=فلكلور', '4=ثقافة', '5=دينبذرة الأساطير الأوقيانوسية', 'آسيا', 'آل سعود', 'آيسلندا', 'أبل', 'أثاث', 'أثينا', 'أحداث جارية', 'أخلاقيات', 'أدب', 'أدب أطفال', 'أدب أمريكي', 'أدب إسباني', 'أدب إنجليزي', 'أدب عربي', 'أدب فرنسي', 'أديان', 'أذربيجان', 'أرثوذكسية', 'أرقام قياسية', 'أرمينيا', 'أستراليا', 'أسلحة', 'أسماك الزينة', 'أعلام', 'أعمال إلكترونية', 'أفريقيا', 'أفغانستان', 'ألاسكا', 'ألبانيا', 'ألعاب', 'ألعاب أولمبية', 'ألعاب القوى', 'ألعاب تقمص أدوار', 'ألعاب فيديو', 'ألمانيا', 'ألمانيا الشرقية', 'ألمانيا النازية', 'ألوان', 'أمازيغ', 'أمريكا اللاتينية', 'أمستردام', 'أمن الحاسوب', 'أندورا', 'أنغولا', 'أنمي ومانغا', 'أنهار', 'أوروبا', 'أوزبكستان', 'أوغندا', 'أوقيانوسيا', 'أوكرانيا', 'أيرلندا', 'أيرلندا الشمالية', 'أيض', 'إباحية', 'إثيوبيا', 'إحصاء', 'إدارة أعمال', 'إريتريا', 'إسبانيا', 'إستونيا', 'إسرائيل', 'إسطنبول', 'إسلام', 'إسلام سياسي', 'إعلام', 'إلحاد', 'إلكترونيات', 'إمبراطورية اليابان', 'إنترنت', 'إنجلترا', 'إندونيسيا', 'إيران', 'إيرانبذرة إ

In [11]:

# =========================================================
# 8) التنبؤ والتقييم
# =========================================================
y_pred = clf.predict(X_test_vec)

acc = accuracy_score(y_test_enc, y_pred)
print("\nAccuracy:", acc)

print("\nClassification Report:\n")
print(
    classification_report(
        y_test_enc,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)


# =========================================================
# 9) تجربة على نص جديد
# =========================================================
def predict_text(text, w2v_model, classifier, label_encoder):
    tokens = tokenize(text)
    vec = document_vector(tokens, w2v_model).reshape(1, -1)
    pred = classifier.predict(vec)[0]
    return label_encoder.inverse_transform([pred])[0]

sample_text = "ضع هنا نصاً جديداً للتجربة"
predicted_class = predict_text(sample_text, w2v_model, clf, label_encoder)
print("\nالصنف المتوقع للنص التجريبي:", predicted_class)


Accuracy: 0.33915900131406046

Classification Report:



ValueError: Number of classes, 758, does not match size of target_names, 759. Try specifying the labels parameter